# Test fixtures
Create reusable test data without sharing mutable state.


In [ ]:
from dataclasses import dataclass

@dataclass
class Task:
    id: int
    title: str

def task_fixture(title: str = "Learn testing") -> Task:
    return Task(id=1, title=title)

print(task_fixture(), task_fixture("Ship API"))


## Polished version
Use a context builder that creates fresh adapters and connected services for every test.


In [ ]:
class MemoryTaskRepository:
    def __init__(self) -> None:
        self.tasks: dict[int, Task] = {}
    def add(self, task: Task) -> None:
        self.tasks[task.id] = task

class TaskService:
    def __init__(self, repository: MemoryTaskRepository) -> None:
        self.repository = repository
    def create(self, title: str) -> Task:
        task = Task(id=len(self.repository.tasks) + 1, title=title)
        self.repository.add(task)
        return task

@dataclass
class TestContext:
    repository: MemoryTaskRepository
    service: TaskService

    @classmethod
    def build(cls) -> "TestContext":
        repository = MemoryTaskRepository()
        return cls(repository, TaskService(repository))

first = TestContext.build()
second = TestContext.build()
first.service.create("Ship API")
assert len(first.repository.tasks) == 1
assert second.repository.tasks == {}
print("fresh state confirmed")
